# Basics &mdash; Cartesian Product, Binary Relations, and Functions

**Concept 11 of the Basics decomposition:** *Cartesian Product, Binary Relations, and Functions*

$A\times B$ is all ordered pairs; a relation is a subset of it; a function is single-valued.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Cartesian-Product-Relations-Functions/Concept-Cartesian-Product-Relations-Functions.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Basics/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


$$A \times B = \{(a,b) : a\in A,\ b\in B\}, \qquad |A\times B| = |A|\cdot|B|$$

A **binary relation** from $A$ to $B$ is **any subset** of $A\times B$. A **function**
is a relation that is **single-valued**: each $a$ appears in at most one pair.

So the chain is: *product* $\supseteq$ *relation* $\supseteq$ *function*, each a
special case of the one before.

This is not decoration. A DFA's $\delta$ is a function $Q\times\Sigma \to Q$; an NFA's
is a function into ${\cal P}(Q)$ &mdash; still single-valued, because the *set* is one
value. That is the precise sense in which an NFA is deterministic over sets, which is
what makes the subset construction work.

## 2. Definitions

### Products, relations, functions

In [ ]:
def cartesian(A, B):
    return {(a, b) for a in A for b in B}

def is_function(R, A):
    seen = {}
    for a, b in R:
        if a in seen and seen[a] != b: return False
        seen[a] = b
    return True

def is_total(R, A):
    return {a for a, _ in R} == set(A)

### Jove's `product`

In [ ]:
A, B = {1, 2, 3}, {'x', 'y'}
print("product(A, B) :", sorted(product(A, B)))

## 3. Tests

$|A\times B| = |A|\cdot|B|$.

In [ ]:
for a, b in [(3, 2), (4, 4), (5, 1), (0, 7)]:
    A, B = set(range(a)), set(range(100, 100 + b))
    p = cartesian(A, B)
    print("  |A|=%d, |B|=%d -> |A x B| = %2d   (product = %d)" % (a, b, len(p), a * b))
    assert len(p) == a * b

Jove's `product` agrees with the definition.

In [ ]:
A, B = {1, 2, 3}, {'x', 'y'}
assert {tuple(t) for t in product(A, B)} == cartesian(A, B)
print("product(A,B) == { (a,b) : a in A, b in B }  ->", True)

**A relation is any subset**; a **function** is single-valued.

In [ ]:
A, B = {1, 2, 3}, {'x', 'y'}
REL = {
 'a function'          : {(1, 'x'), (2, 'y'), (3, 'x')},
 'not single-valued'   : {(1, 'x'), (1, 'y'), (2, 'x')},
 'partial function'    : {(1, 'x'), (3, 'y')},
 'the empty relation'  : set(),
 'the full product'    : cartesian(A, B),
}
print("%-22s %-10s %-10s" % ("relation", "function?", "total?"))
for name, R in REL.items():
    print("%-22s %-10s %-10s" % (name, is_function(R, A), is_total(R, A)))
assert is_function(REL['a function'], A)
assert not is_function(REL['not single-valued'], A)
assert is_function(REL['partial function'], A) and not is_total(REL['partial function'], A)

**A DFA's $\delta$ is a function on a product.**

In [ ]:
D = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> IF
Od : 0 -> IF
Od : 1 -> Od
''')
dom = cartesian(D["Q"], D["Sigma"])
print("Q x Sigma :", sorted(dom))
print("|Delta|   :", len(D["Delta"]), " |Q x Sigma| :", len(dom))
assert set(D["Delta"].keys()) == dom
assert len(D["Delta"]) == len(D["Q"]) * len(D["Sigma"])
print("\ndelta is TOTAL on Q x Sigma -- every pair has exactly one image.")

**An NFA's $\delta$ is still single-valued** &mdash; its values are sets.

In [ ]:
N = md2mc('''NFA
I : 0 -> I
I : 0 -> F
''')
print("NFA Delta :", dict(N["Delta"]))
for k, v in N["Delta"].items():
    assert isinstance(v, set)
print("\nEvery value is ONE set.  The nondeterminism is inside the value,")
print("not in the function being multi-valued -- which is exactly why the")
print("subset construction (Chapter 7) can treat a set as a single state.")

## 4. Exercises


1. How many binary relations are there from a 3-set to a 2-set? How many functions?
2. Is $A\times B = B\times A$? When?
3. Write a PDA's $\Delta$ as a subset of a product. What are the factors?

In [ ]:
# Your work for the exercises above.